# Remote model evaluation on the validation split

**Code has significant portions generated by CLAUDE**

`cross_val_evaluation.ipynb` evaluates models across a 10-fold cross-validation
over the *entire* dataset (train+val+test merged), which means ~1091 requests
per model. Running `qwen3-30b-a3b-instruct-2507` against the remote
OpenAI-compatible inference server at that volume hit API rate/quota limits
partway through and lost all progress, since predictions were only persisted
once a whole fold finished.

This notebook instead:
- Evaluates on the original **validation split only** (`open_data/bzkopen_addresses_val.csv`,
  152 addresses) rather than the merged/cross-validated dataset, using the
  original **train split** (`open_data/bzkopen_addresses_train.csv`) as the
  few-shot example pool — no custom folds, no re-training of the NER model
  (`models/ner_bzk` was already trained on this exact train/val split).
- Uses the same prompting configuration as `Qwen3.5-9B-best-from-optuna` in
  `cross_val_evaluation.ipynb` (same prompt template, example-matching
  strategy, and hyperparameters).
- Persists predictions to disk after every small chunk of addresses, and
  skips already-cached addresses on rerun, so hitting a rate limit only
  costs the in-flight chunk instead of the whole run.

## Setup

In [1]:
import json
from collections import OrderedDict
from pathlib import Path
import time

import pandas as pd
import torch
from tqdm.auto import tqdm

import modules.llms as llm_parsers
from modules.utils import compare_preds, format_time

REQUIRED_ENTITIES = [
    "HouseNumber",
    "StreetName",
    "City",
    "Country"
]

class RateLimitKeeper:
    def __init__(self, max_calls: int, time_window: int, silent : bool = False):
        self.max_calls = max_calls
        self.time_window = time_window
        self.calls_made = 0
        self.start_time = time.monotonic()
        self.total_sleep_time = 0.0
        self.silent = silent

    def rate_limit_call(self):
        if self.calls_made == 0:
            self.start_time = time.monotonic()
            self.calls_made = 1
        elif self.calls_made >= self.max_calls:
            elapsed_time = time.monotonic() - self.start_time
            if elapsed_time < self.time_window:
                sleep_time = self.time_window - elapsed_time
                if not self.silent:
                    print(f"Rate limit of {self.max_calls} with time window{format_time(self.time_window)} reached. "
                          f"Sleeping for {format_time(sleep_time)}.")
                self.total_sleep_time += sleep_time
                time.sleep(sleep_time)
            self.start_time = time.monotonic()
            self.calls_made = 1
        else:
            self.calls_made += 1

    def reset_total_sleep_time(self):
        self.total_sleep_time = 0.0

class CompoundRateLimitKeeper:
    def __init__(self, rate_limiters : list[RateLimitKeeper]):
        self.rate_limiters = rate_limiters

    @property
    def total_sleep_time(self):
        return sum(rl.total_sleep_time for rl in self.rate_limiters)

    def rate_limit_call(self):
        for rl in self.rate_limiters:
            rl.rate_limit_call()

    def reset_total_sleep_time(self):
        for rl in self.rate_limiters:
            rl.reset_total_sleep_time()

rate_limit_keeper = CompoundRateLimitKeeper([
    RateLimitKeeper(max_calls=10, time_window=60),
    RateLimitKeeper(max_calls=200, time_window=60*60),
    RateLimitKeeper(max_calls=400, time_window=60*60*24),
])

In [2]:
# Only used for the local embedding model in the example-matching strategy;
# the address-parsing model itself runs on the remote inference server.
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Embedding device: {device}")

Embedding device: cuda


## Load dataset

Unlike `cross_val_evaluation.ipynb`, we keep the original train/val split
instead of merging and re-splitting into cross-validation folds: the train
split is the few-shot example pool, the val split is what gets evaluated.

In [3]:
csv_read_args = dict(keep_default_na=False, dtype=str, na_values=[""])

train_data = pd.read_csv("open_data/bzkopen_addresses_train.csv", **csv_read_args)
val_data = pd.read_csv("open_data/bzkopen_addresses_val.csv", **csv_read_args)

print(f"train: {len(train_data)} addresses, val: {len(val_data)} addresses")
display(val_data.sample(5))

train: 771 addresses, val: 152 addresses


,card_id,field,FullAddress,UnitNumber,HouseNumber,StreetName,Neighborhood,City,District,Region,State,Country,PostalCode
150,val_76,ApplicantBirthPlace,Rum.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Rum.,NaN
11,val_7,ApplicantCurrentAddress,"New York N.Y. USA, 210 West 70 Str. Apt. 301",Apt. 301,210,West 70 Str.,NaN,New York,NaN,NaN,N.Y.,USA,NaN
14,val_9,ApplicantBirthPlace,Haspe-Hagen,NaN,NaN,NaN,Haspe,Hagen,NaN,NaN,NaN,NaN,NaN
68,val_37,ApplicantCurrentAddress,Ittlingen Krs. Sinsheim/Elsenz,NaN,NaN,NaN,NaN,Ittlingen,Sinsheim,Elsenz,NaN,NaN,NaN
109,val_56,VictimBirthPlace,Hagen-Haspe,NaN,NaN,NaN,Haspe,Hagen,NaN,NaN,NaN,NaN,NaN


## Prompting configuration

Same prompt template, supported entities, example count, embedding model and
similarity threshold as `Qwen3.5-9B-best-from-optuna` in
`cross_val_evaluation.ipynb`.

In [4]:
prompt_qwen_optuna_best = llm_parsers.JsonDictPromptTemplate(Path("prompts/optuna_best/best_qwen_prompt.txt").read_text())

supported_entities = ["HouseNumber", "StreetName", "Neighborhood", "City", "Country"]
n_examples = 15
embedding_model = "all-MiniLM-L6-v2"
similarity_threshold = 0.35

# Already trained on this exact train/val split (see modules/train_ner.py),
# so it can be reused as-is instead of retraining per fold.
ner_model_dir = "models/ner_bzk"

## Build the example-matching strategy

Same hybrid NER-pattern + embedding-similarity strategy as the cross-val
notebook, built once from the train split (no per-fold loop needed since
there is only one split here).

In [5]:
pattern_similarity = llm_parsers.NERPatternSimilarExamples(
    example_addresses=train_data['FullAddress'].reset_index(drop=True),
    example_labels=train_data,
    labels_to_include=supported_entities,
    num_examples=n_examples,
    model_dir=ner_model_dir
)
embedding_similarity = llm_parsers.SimilarExamples(
    embedding_model=embedding_model,
    example_addresses=train_data['FullAddress'].reset_index(drop=True),
    example_labels=train_data,
    labels_to_include=supported_entities,
    num_examples=n_examples,
    similarity_threshold=similarity_threshold,
    device=device
)
hybrid_similarity = llm_parsers.HybridSimilarExamples(
    pattern_strategy=pattern_similarity,
    embedding_strategy=embedding_similarity,
    num_examples=n_examples,
    pool_size=n_examples
)

Computing NER patterns for 771 training examples...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Remote model

`max_workers` and `chunk_size` (below) are kept low on purpose to stay under
the inference server's rate limits; `max_retries` lets the OpenAI client
retry transient/rate-limit errors with backoff before giving up.

## Run on the validation split, with resumable caching

Predictions are cached to disk per-address and reloaded on rerun, so if a
chunk fails (e.g. rate limit exceeded), progress up to that point is kept and
re-running the cell picks up where it left off instead of re-querying
everything.

In [6]:
def run_validation_split(model : llm_parsers.RemoteAddressParsingModel, val_data, preds_file):
    addresses = val_data['FullAddress'].tolist()
    if preds_file.exists():
        with open(preds_file, "r", encoding="utf-8") as f:
            cache = json.load(f)
    else:
        cache = {}
        try:
            rate_limit_keeper.reset_total_sleep_time()
            start = time.monotonic()
            preds = model.parse_addresses(addresses)
            end = time.monotonic()
            cache["actual_deltatime"] = end - start
            cache["deltatime"] = end - start - rate_limit_keeper.total_sleep_time
            cache["preds"] = preds
            with open(preds_file, "w", encoding="utf-8") as f:
                json.dump(cache, f, indent=4, ensure_ascii=False)
        except Exception as e:
            print(f"Error occurred while parsing addresses: {e}")
            return pd.DataFrame(index=val_data.index)  # Return an empty DataFrame on error
    preds_df = pd.DataFrame(cache["preds"], index=val_data.index)
    actual_deltatime = cache.get("actual_deltatime")
    deltatime_net = cache.get("deltatime")
    if actual_deltatime is not None and deltatime_net is not None:
        sleeptime = actual_deltatime - deltatime_net
        print(f"Total time: {format_time(actual_deltatime)}, net time: {format_time(deltatime_net)}, sleep time: {format_time(sleeptime)}")
        print(f"Expected rate: {len(addresses) / deltatime_net:.2f} addresses/s")
        time_per_batch = deltatime_net / (len(addresses)/10)
        print(f"Time per batch of 10 addresses: {time_per_batch:.2f}s")
        print(f"Expected time for 4 394 539 addresses:\n\t{format_time(deltatime_net * 4_394_539 / len(addresses))}")
    return preds_df

results_for_all_models = OrderedDict()

## Results

In [7]:
model_name = "qwen3-30b-a3b-instruct-2507"
def eval_model(model_name):
    model = llm_parsers.RemoteAddressParsingModel(
        model_name=model_name,
        rate_limit_keeper=rate_limit_keeper,
        credentials_path="ai_api_credentials.json",
        example_strategy=hybrid_similarity,
        prompt=prompt_qwen_optuna_best,
        extra_client_kwargs={"max_retries": 5}
    )

    preds_path = Path("experiments_data") / "remote_val" / model_name
    preds_file = preds_path / "preds.json"
    preds_file.parent.mkdir(parents=True, exist_ok=True)
    preds_df = run_validation_split(model, val_data, preds_file)
    required_metrics = pd.Series(
        compare_preds(preds_df, val_data, target_columns=REQUIRED_ENTITIES),
        name="Country/City/Street/House"
    )
    results_for_all_models[model.model_name] = required_metrics
    print(f"Overall results for {model_name}")
    display(required_metrics)

    specific_metrics = OrderedDict()
    for entity in supported_entities:
        specific_metrics[entity] = compare_preds(preds_df, val_data, target_columns=[entity])
    print(f"Specific results for {model_name}")
    display(pd.DataFrame(specific_metrics))

    BZK_ADDRESS_FIELDS = [
        'ApplicantCurrentAddress',
        'VictimBirthPlace',
        'VictimCurrentAddress',
        'ApplicantBirthPlace',
        'VictimDeathPlace'
    ]

    field_metrics = OrderedDict()
    for field in BZK_ADDRESS_FIELDS:
        mask = val_data['field'] == field
        field_metrics[field] = compare_preds(preds_df[mask], val_data[mask], target_columns=REQUIRED_ENTITIES)
    print(f"Field-specific results for {model_name}")
    display(pd.DataFrame(field_metrics))

In [8]:
try:
    eval_model("qwen3-30b-a3b-instruct-2507")
except Exception as e:
    print(f"Error occurred while evaluating model {model_name}: {e}")

Overall results for qwen3-30b-a3b-instruct-2507


accuracy                     0.955592
precision                    0.918495
recall                       0.942122
f1                           0.930159
accuracy_with_tol_1          0.955592
accuracy_with_tol_2          0.960526
accuracy_with_tol_3          0.972039
accuracy_with_tol_4          0.973684
average_levenshtein          0.345395
average_similarity           0.965994
average_levenshtein_match    0.351171
average_similarity_match     0.982148
no_match_rate                0.016447
Name: Country/City/Street/House, dtype: float64

Specific results for qwen3-30b-a3b-instruct-2507


,HouseNumber,StreetName,Neighborhood,City,Country
accuracy,0.967105,0.973684,0.921053,0.921053,0.960526
precision,0.918033,0.937500,0.647059,0.925170,0.872340
recall,0.918033,0.967742,0.611111,0.931507,0.976190
f1,0.918033,0.952381,0.628571,0.928328,0.921348
accuracy_with_tol_1,0.967105,0.973684,0.921053,0.921053,0.960526
accuracy_with_tol_2,0.980263,0.973684,0.934211,0.921053,0.967105
accuracy_with_tol_3,0.993421,0.986842,0.934211,0.934211,0.973684
accuracy_with_tol_4,1.000000,0.986842,0.934211,0.934211,0.973684
average_levenshtein,0.092105,0.269737,0.677632,0.703947,0.315789
average_similarity,0.979308,0.980263,0.925282,0.938763,0.965643


Field-specific results for qwen3-30b-a3b-instruct-2507


,ApplicantCurrentAddress,VictimBirthPlace,VictimCurrentAddress,ApplicantBirthPlace,VictimDeathPlace
accuracy,0.906863,1.0,0.947368,0.986111,0.958333
precision,0.889571,1.0,0.928571,0.956522,0.800000
recall,0.917722,1.0,0.945455,0.970588,1.000000
f1,0.903427,1.0,0.936937,0.963504,0.888889
accuracy_with_tol_1,0.906863,1.0,0.947368,0.986111,0.958333
accuracy_with_tol_2,0.916667,1.0,0.947368,0.990741,0.958333
accuracy_with_tol_3,0.950980,1.0,0.947368,0.990741,0.958333
accuracy_with_tol_4,0.950980,1.0,0.960526,0.990741,0.958333
average_levenshtein,0.696078,0.0,0.486842,0.097222,0.416667
average_similarity,0.930857,1.0,0.954302,0.990291,0.958333


In [9]:
try:
    eval_model("gemma-4-31b-it")
except Exception as e:
    print(f"Error occurred while evaluating model gemma-4-31b-it: {e}")

Total time: 0:15:31, net time: 0:00:31, sleep time: 0:15:00
Expected rate: 4.87 addresses/s
Time per batch of 10 addresses: 2.05s
Expected time for 4 394 539 addresses:
	10 days, 10:36:10
Overall results for gemma-4-31b-it


accuracy                     0.967105
precision                    0.951299
recall                       0.942122
f1                           0.946688
accuracy_with_tol_1          0.967105
accuracy_with_tol_2          0.972039
accuracy_with_tol_3          0.980263
accuracy_with_tol_4          0.983553
average_levenshtein          0.241776
average_similarity           0.977367
average_levenshtein_match    0.244592
average_similarity_match     0.988750
no_match_rate                0.011513
Name: Country/City/Street/House, dtype: float64

Specific results for gemma-4-31b-it


,HouseNumber,StreetName,Neighborhood,City,Country
accuracy,0.967105,0.947368,0.927632,0.967105,0.986842
precision,0.933333,0.932203,0.578947,0.965986,0.952381
recall,0.918033,0.887097,0.611111,0.972603,0.952381
f1,0.925620,0.909091,0.594595,0.969283,0.952381
accuracy_with_tol_1,0.967105,0.947368,0.927632,0.967105,0.986842
accuracy_with_tol_2,0.980263,0.947368,0.947368,0.967105,0.993421
accuracy_with_tol_3,0.993421,0.960526,0.947368,0.973684,0.993421
accuracy_with_tol_4,1.000000,0.960526,0.947368,0.980263,0.993421
average_levenshtein,0.092105,0.585526,0.559211,0.230263,0.059211
average_similarity,0.977992,0.957199,0.938794,0.978469,0.995806


Field-specific results for gemma-4-31b-it


,ApplicantCurrentAddress,VictimBirthPlace,VictimCurrentAddress,ApplicantBirthPlace,VictimDeathPlace
accuracy,0.926471,1.0,0.960526,0.990741,1.0
precision,0.929487,1.0,0.962963,0.970588,1.0
recall,0.917722,1.0,0.945455,0.970588,1.0
f1,0.923567,1.0,0.954128,0.970588,1.0
accuracy_with_tol_1,0.926471,1.0,0.960526,0.990741,1.0
accuracy_with_tol_2,0.936275,1.0,0.960526,0.995370,1.0
accuracy_with_tol_3,0.955882,1.0,0.973684,0.995370,1.0
accuracy_with_tol_4,0.965686,1.0,0.973684,0.995370,1.0
average_levenshtein,0.534314,0.0,0.381579,0.041667,0.0
average_similarity,0.945166,1.0,0.974507,0.997049,1.0


In [10]:
try:
    eval_model("qwen3-coder-30b-a3b-instruct")
except Exception as e:
    print(f"Error occurred while evaluating model qwen3-coder-30b-a3b-instruct: {e}")   

Total time: 0:15:12, net time: 0:00:12, sleep time: 0:15:00
Expected rate: 12.30 addresses/s
Time per batch of 10 addresses: 0.81s
Expected time for 4 394 539 addresses:
	4 days, 3:13:28
Overall results for qwen3-coder-30b-a3b-instruct


accuracy                     0.000000
precision                    0.000000
recall                       0.000000
f1                           0.000000
accuracy_with_tol_1          0.000000
accuracy_with_tol_2          0.000000
accuracy_with_tol_3          0.000000
accuracy_with_tol_4          0.000000
average_levenshtein          3.891447
average_similarity           0.000000
average_levenshtein_match    0.000000
average_similarity_match     0.000000
no_match_rate                1.000000
Name: Country/City/Street/House, dtype: float64

Specific results for qwen3-coder-30b-a3b-instruct


,HouseNumber,StreetName,Neighborhood,City,Country
accuracy,0.000000,0.000000,0.000000,0.000000,0.000000
precision,0.000000,0.000000,0.000000,0.000000,0.000000
recall,0.000000,0.000000,0.000000,0.000000,0.000000
f1,0.000000,0.000000,0.000000,0.000000,0.000000
accuracy_with_tol_1,0.000000,0.000000,0.000000,0.000000,0.000000
accuracy_with_tol_2,0.000000,0.000000,0.000000,0.000000,0.000000
accuracy_with_tol_3,0.000000,0.000000,0.000000,0.000000,0.000000
accuracy_with_tol_4,0.000000,0.000000,0.000000,0.000000,0.000000
average_levenshtein,0.921053,4.980263,1.144737,8.131579,1.532895
average_similarity,0.000000,0.000000,0.000000,0.000000,0.000000


Field-specific results for qwen3-coder-30b-a3b-instruct


,ApplicantCurrentAddress,VictimBirthPlace,VictimCurrentAddress,ApplicantBirthPlace,VictimDeathPlace
accuracy,0.000000,0.000000,0.000000,0.000000,0.000000
precision,0.000000,0.000000,0.000000,0.000000,0.000000
recall,0.000000,0.000000,0.000000,0.000000,0.000000
f1,0.000000,0.000000,0.000000,0.000000,0.000000
accuracy_with_tol_1,0.000000,0.000000,0.000000,0.000000,0.000000
accuracy_with_tol_2,0.000000,0.000000,0.000000,0.000000,0.000000
accuracy_with_tol_3,0.000000,0.000000,0.000000,0.000000,0.000000
accuracy_with_tol_4,0.000000,0.000000,0.000000,0.000000,0.000000
average_levenshtein,5.666667,2.454545,5.657895,2.462963,1.333333
average_similarity,0.000000,0.000000,0.000000,0.000000,0.000000


In [11]:
try:
    eval_model("qwen3-omni-30b-a3b-instruct")
except Exception as e:
    print(f"Error occurred while evaluating model qwen3-omni-30b-a3b-instruct: {e}")

Total time: 0:54:01, net time: 0:00:13, sleep time: 0:53:48
Expected rate: 11.72 addresses/s
Time per batch of 10 addresses: 0.85s
Expected time for 4 394 539 addresses:
	4 days, 8:07:21
Overall results for qwen3-omni-30b-a3b-instruct


accuracy                     0.794408
precision                    0.909091
recall                       0.610932
f1                           0.730769
accuracy_with_tol_1          0.809211
accuracy_with_tol_2          0.822368
accuracy_with_tol_3          0.843750
accuracy_with_tol_4          0.848684
average_levenshtein          1.542763
average_similarity           0.802429
average_levenshtein_match    1.887324
average_similarity_match     0.981643
no_match_rate                0.182566
Name: Country/City/Street/House, dtype: float64

Specific results for qwen3-omni-30b-a3b-instruct


,HouseNumber,StreetName,Neighborhood,City,Country
accuracy,0.842105,0.822368,0.875000,0.611842,0.901316
precision,0.948718,0.923077,0.454545,0.881188,0.933333
recall,0.606557,0.580645,0.277778,0.609589,0.666667
f1,0.740000,0.712871,0.344828,0.720648,0.777778
accuracy_with_tol_1,0.901316,0.822368,0.875000,0.611842,0.901316
accuracy_with_tol_2,0.947368,0.822368,0.894737,0.611842,0.907895
accuracy_with_tol_3,0.986842,0.828947,0.894737,0.631579,0.927632
accuracy_with_tol_4,0.986842,0.828947,0.894737,0.644737,0.934211
average_levenshtein,0.361842,2.144737,1.164474,3.111842,0.552632
average_similarity,0.849154,0.826754,0.875000,0.627228,0.906579


Field-specific results for qwen3-omni-30b-a3b-instruct


,ApplicantCurrentAddress,VictimBirthPlace,VictimCurrentAddress,ApplicantBirthPlace,VictimDeathPlace
accuracy,0.647059,0.931818,0.710526,0.888889,0.958333
precision,0.880000,1.000000,0.894737,0.956522,0.800000
recall,0.556962,0.769231,0.618182,0.647059,1.000000
f1,0.682171,0.869565,0.731183,0.771930,0.888889
accuracy_with_tol_1,0.686275,0.931818,0.723684,0.888889,0.958333
accuracy_with_tol_2,0.710784,0.931818,0.750000,0.893519,0.958333
accuracy_with_tol_3,0.764706,0.931818,0.776316,0.893519,0.958333
accuracy_with_tol_4,0.764706,0.943182,0.789474,0.898148,0.958333
average_levenshtein,2.519608,0.590909,2.473684,0.805556,0.416667
average_similarity,0.664011,0.931818,0.717021,0.893171,0.958333


In [12]:

try:
    eval_model("Mistral-Medium-3.5-128B")
except Exception as e:
    print(f"Error occurred while evaluating model Mistral-Medium-3.5-128B: {e}")

Total time: 22:54:59, net time: 0:00:12, sleep time: 22:54:47
Expected rate: 12.33 addresses/s
Time per batch of 10 addresses: 0.81s
Expected time for 4 394 539 addresses:
	4 days, 3:01:32
Overall results for Mistral-Medium-3.5-128B


accuracy                     0.000000
precision                    0.000000
recall                       0.000000
f1                           0.000000
accuracy_with_tol_1          0.000000
accuracy_with_tol_2          0.000000
accuracy_with_tol_3          0.000000
accuracy_with_tol_4          0.000000
average_levenshtein          3.891447
average_similarity           0.000000
average_levenshtein_match    0.000000
average_similarity_match     0.000000
no_match_rate                1.000000
Name: Country/City/Street/House, dtype: float64

Specific results for Mistral-Medium-3.5-128B


,HouseNumber,StreetName,Neighborhood,City,Country
accuracy,0.000000,0.000000,0.000000,0.000000,0.000000
precision,0.000000,0.000000,0.000000,0.000000,0.000000
recall,0.000000,0.000000,0.000000,0.000000,0.000000
f1,0.000000,0.000000,0.000000,0.000000,0.000000
accuracy_with_tol_1,0.000000,0.000000,0.000000,0.000000,0.000000
accuracy_with_tol_2,0.000000,0.000000,0.000000,0.000000,0.000000
accuracy_with_tol_3,0.000000,0.000000,0.000000,0.000000,0.000000
accuracy_with_tol_4,0.000000,0.000000,0.000000,0.000000,0.000000
average_levenshtein,0.921053,4.980263,1.144737,8.131579,1.532895
average_similarity,0.000000,0.000000,0.000000,0.000000,0.000000


Field-specific results for Mistral-Medium-3.5-128B


,ApplicantCurrentAddress,VictimBirthPlace,VictimCurrentAddress,ApplicantBirthPlace,VictimDeathPlace
accuracy,0.000000,0.000000,0.000000,0.000000,0.000000
precision,0.000000,0.000000,0.000000,0.000000,0.000000
recall,0.000000,0.000000,0.000000,0.000000,0.000000
f1,0.000000,0.000000,0.000000,0.000000,0.000000
accuracy_with_tol_1,0.000000,0.000000,0.000000,0.000000,0.000000
accuracy_with_tol_2,0.000000,0.000000,0.000000,0.000000,0.000000
accuracy_with_tol_3,0.000000,0.000000,0.000000,0.000000,0.000000
accuracy_with_tol_4,0.000000,0.000000,0.000000,0.000000,0.000000
average_levenshtein,5.666667,2.454545,5.657895,2.462963,1.333333
average_similarity,0.000000,0.000000,0.000000,0.000000,0.000000


In [13]:
display(pd.DataFrame(results_for_all_models))

,qwen3-30b-a3b-instruct-2507,gemma-4-31b-it,qwen3-coder-30b-a3b-instruct,qwen3-omni-30b-a3b-instruct,Mistral-Medium-3.5-128B
accuracy,0.955592,0.967105,0.000000,0.794408,0.000000
precision,0.918495,0.951299,0.000000,0.909091,0.000000
recall,0.942122,0.942122,0.000000,0.610932,0.000000
f1,0.930159,0.946688,0.000000,0.730769,0.000000
accuracy_with_tol_1,0.955592,0.967105,0.000000,0.809211,0.000000
accuracy_with_tol_2,0.960526,0.972039,0.000000,0.822368,0.000000
accuracy_with_tol_3,0.972039,0.980263,0.000000,0.843750,0.000000
accuracy_with_tol_4,0.973684,0.983553,0.000000,0.848684,0.000000
average_levenshtein,0.345395,0.241776,3.891447,1.542763,3.891447
average_similarity,0.965994,0.977367,0.000000,0.802429,0.000000
